<div style="text-align:left;">
  <a href="https://code213.tech/" target="_blank">
    <img src="../images/code213.PNG" alt="Code213 Logo" width="200"/>
  </a>

<div style="text-align:left;">
  <a href="https://huggingface.co/hellosara" target="_blank">
    <img src="https://huggingface.co/front/assets/huggingface_logo-noborder.svg" alt="HF Logo" width="50"/>
  </a>
  <p><em>Prepared by Latreche Sara</em></p>
</div>

<center><font size=6><b>Deep RL Lab: FrozenLake-v1 with Q-Learning</b></font></center>

<div style="text-align:center;">
    <img src="https://gymnasium.farama.org/_images/frozen_lake.gif" width="450" style="border-radius: 10px; box-shadow: 0px 4px 15px rgba(0,0,0,0.2);"/>
</div>

***

**Author**: Latreche Sara  
**Date**: February 2026  
**Project Overview**: In this lab, we trained a **Reinforcement Learning** agent to navigate a grid world and reach a goal using the **Q-Learning** algorithm.
Q-Learning is a **value-based** off-policy method that learns the optimal actions by maintaining a Q-Table, mapping every state-action pair to its expected future reward.

We focused on:  
- Initializing the **FrozenLake-v1** environment (4x4, non-slippery).  
- Creating and managing a **Q-Table** to store state-action values.  
- Implementing an **Epsilon-Greedy** strategy to balance exploration and exploitation.  
- Achieving a perfect evaluation score (**1.00** success rate).  
- Uploading the trained Q-Table to the **Hugging Face Hub**.

---

## Table of Contents  

- [1 - Environment Setup & Exploration](#1)  
- [2 - Q-Table Initialization](#2)  
- [3 - Epsilon-Greedy Policy](#3)  
- [4 - Training the Q-Learning Agent](#4)  
- [5 - Evaluation & Result Analysis](#5)  
- [6 - Model Deployment to the Hub](#6)

---

### 💡 Why Q-Learning?
Unlike the Lunar Lander project, which required a Neural Network (PPO) to handle continuous coordinates, **FrozenLake** has a finite number of states (16). This allows us to use a **Q-Table**, which acts as a lookup table where the agent can directly "read" the best move to make in any given position.

# 1 - Environment Setup & Exploration ❄️ <a id='1'></a>

### 👨‍🏫 Student Insight: What is Frozen Lake?
In this environment, the agent must cross a 4x4 grid. The challenge is the **Sparse Reward**: the agent receives a `0` for every step it takes and only receives a `+1` if it successfully reaches the Goal. If it falls into a Hole, the episode ends with a reward of `0`. This means the agent must "stumble" upon the goal by accident before it can start learning which moves are good.

In [1]:
import pickle
import numpy as np
import gymnasium as gym
import random
import imageio
import os
from tqdm.notebook import tqdm

# Create the FrozenLake-v1 environment
# map_name="4x4": A small grid
# is_slippery=False: Deterministic (Agent moves exactly where told)
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="rgb_array")

In [2]:
# We create our environment with gym.make("<name_of_the_environment>")- `is_slippery=False`: The agent always moves in the intended direction due to the non-slippery nature of the frozen lake (deterministic).
print("_____OBSERVATION SPACE_____ \n")
print("Observation Space", env.observation_space)
print("Sample observation", env.observation_space.sample())  # Get a random observation

_____OBSERVATION SPACE_____ 

Observation Space Discrete(16)
Sample observation 9


We see with Observation Space Shape Discrete(16) that the observation is an integer representing the agent’s current position as current_row * ncols + current_col (where both the row and col start at 0).

For example, the goal position in the 4x4 map can be calculated as follows: 3 * 4 + 3 = 15. The number of possible observations is dependent on the size of the map. For example, the 4x4 map has 16 possible observations.

For instance, this is what state = 0 looks like:



In [3]:
print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample())  # Take a random action


 _____ACTION SPACE_____ 

Action Space Shape 4
Action Space Sample 1


The action space (the set of possible actions the agent can take) is discrete with 4 actions available 🎮:

0: GO LEFT
1: GO DOWN
2: GO RIGHT
3: GO UP
Reward function 💰:

Reach goal: +1
Reach hole: 0
Reach frozen: 0

Create and Initialize the Q-table

It’s time to initialize our Q-table! To know how many rows (states) and columns (actions) to use, we need to know the action and observation space. We already know their values from before, but we’ll want to obtain them programmatically so that our algorithm generalizes for different environments. Gym provides us a way to do that: env.action_space.n and env.observation_space.n

In [4]:
state_space = env.observation_space.n
action_space = env.action_space.n

In [5]:
# state_space =
print("There are ", state_space, " possible states")

#action_space =
print("There are ", action_space, " possible actions")

There are  16  possible states
There are  4  possible actions


## 2 - Q-Table Initialization 🗄️ <a id='2'></a>
👨‍🏫 Student Insight: The Memory Matrix
Because our agent doesn't have a "brain" (Neural Network) yet, it uses a Lookup Table.

Imagine a spreadsheet where every row is a tile on the map (0-15).

Every column is a direction (0-3).

The numbers inside represent the "Quality" (Q-value) of taking that action in that state.

In [6]:
def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))

Qtable_frozenlake = initialize_q_table(state_space, action_space)
print(f"Q-Table Shape: {Qtable_frozenlake.shape}")

Q-Table Shape: (16, 4)


## 3 - Q-Learning Agent: The Policy 🤖 <a id='3'></a>
👨‍🏫 Student Insight: The Epsilon-Greedy Strategy

How does an agent learn?If it only Exploits (takes the best known move), it might never find the Goal.If it only Explores (moves randomly), it will never become efficient.We use Epsilon ($\epsilon$) to start with 100% exploration and slowly transition to 95% exploitation as the agent becomes smarter.

In [7]:
def greedy_policy(Qtable, state):
    # Exploitation: take the action with the highest value
    return np.argmax(Qtable[state][:])

def epsilon_greedy_policy(Qtable, state, epsilon):
    random_num = random.uniform(0, 1)
    if random_num > epsilon:
        # Exploitation
        return greedy_policy(Qtable, state)
    else:
        # Exploration: random action
        return env.action_space.sample()

## 4 - Training the Agent (The Bellman Equation) 🏋️ <a id='4'></a>

👨‍🏫 Student Insight: Learning from Mistakes

The core of Q-Learning is the Bellman Equation. Every time the agent takes a step, it compares what it thought the reward would be vs. what it actually received. It then updates the Q-Table to be slightly more accurate for next time.

In [8]:
# Training parameters
n_training_episodes = 10000  # Total training episodes
learning_rate = 0.7  # Learning rate

# Evaluation parameters
n_eval_episodes = 100  # Total number of test episodes

# Environment parameters
env_id = "FrozenLake-v1"  # Name of the environment
max_steps = 99  # Max steps per episode
gamma = 0.95  # Discounting rate
eval_seed = []  # The evaluation seed of the environment

# Exploration parameters
max_epsilon = 1.0  # Exploration probability at start
min_epsilon = 0.05  # Minimum exploration probability
decay_rate = 0.0005  # Exponential decay rate for exploration prob

For episode in the total of training episodes:

Reduce epsilon (since we need less and less exploration)
Reset the environment

  For step in max timesteps:
    Choose the action At using epsilon greedy policy
    Take the action (a) and observe the outcome state(s') and reward (r)
    Update the Q-value Q(s,a) using Bellman equation Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
    If done, finish the episode
    Our next state is the new state

In [9]:
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
    for episode in tqdm(range(n_training_episodes)):
        # Reduce epsilon (because we need less and less exploration)
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
        # Reset the environment
        state, info = env.reset()
        step = 0
        terminated = False
        truncated = False

        # repeat
        for step in range(max_steps):
            # Choose the action At using epsilon greedy policy
            action = epsilon_greedy_policy(Qtable, state, epsilon)

            # Take action At and observe Rt+1 and St+1
            # Take the action (a) and observe the outcome state(s') and reward (r)
            new_state, reward, terminated, truncated, info = env.step(action)

            # Update Q(s,a):= Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
            Qtable[state][action] = Qtable[state][action] + learning_rate * (
                reward + gamma * np.max(Qtable[new_state]) - Qtable[state][action]
            )

            # If terminated or truncated finish the episode
            if terminated or truncated:
                break

            # Our next state is the new state
            state = new_state
    return Qtable

Let’s see what our Q-Learning table looks like now 👀


In [10]:
Qtable_frozenlake = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_frozenlake)


  0%|          | 0/10000 [00:00<?, ?it/s]

In [11]:
Qtable_frozenlake


array([[0.73509189, 0.77378094, 0.77378094, 0.73509189],
       [0.73509189, 0.        , 0.81450625, 0.77378094],
       [0.77378094, 0.857375  , 0.77378094, 0.81450625],
       [0.81450625, 0.        , 0.77378094, 0.77378094],
       [0.77378094, 0.81450625, 0.        , 0.73509189],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.9025    , 0.        , 0.81450625],
       [0.        , 0.        , 0.        , 0.        ],
       [0.81450625, 0.        , 0.857375  , 0.77378094],
       [0.81450625, 0.9025    , 0.9025    , 0.        ],
       [0.857375  , 0.95      , 0.        , 0.857375  ],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.9025    , 0.95      , 0.857375  ],
       [0.9025    , 0.95      , 1.        , 0.9025    ],
       [0.        , 0.        , 0.        , 0.        ]])

The evaluation method 📝
We defined the evaluation method that we’re going to use to test our Q-Learning agent.

## 5 - Evaluation & Result Analysis 📈 <a id='5'></a>

👨‍🏫 Student Insight: Testing the "Brain"

During evaluation, we set Epsilon to 0 (No exploration). We want to see if the agent can reach the Goal 100% of the time using only its learned Q-Table. In a non-slippery lake, a score of 1.00 is a perfect pass!

In [12]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seed):
    """
    Evaluate the agent for ``n_eval_episodes`` episodes and returns average reward and std of reward.
    :param env: The evaluation environment
    :param n_eval_episodes: Number of episode to evaluate the agent
    :param Q: The Q-table
    :param seed: The evaluation seed array (for taxi-v3)
    """
    episode_rewards = []
    for episode in tqdm(range(n_eval_episodes)):
        if seed:
            state, info = env.reset(seed=seed[episode])
        else:
            state, info = env.reset()
        step = 0
        truncated = False
        terminated = False
        total_rewards_ep = 0

        for step in range(max_steps):
            # Take the action (index) that have the maximum expected future reward given that state
            action = greedy_policy(Q, state)
            new_state, reward, terminated, truncated, info = env.step(action)
            total_rewards_ep += reward

            if terminated or truncated:
                break
            state = new_state
        episode_rewards.append(total_rewards_ep)
    mean_reward = np.mean(episode_rewards)
    std_reward = np.std(episode_rewards)

    return mean_reward, std_reward

Evaluate our Q-Learning agent 📈
Usually, you should have a mean reward of 1.0
The environment is relatively easy since the state space is really small (16). What you can try to do is to replace it with the slippery version, which introduces stochasticity, making the environment more complex.

In [13]:
# Evaluate our Agent
mean_reward, std_reward = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_frozenlake, eval_seed)
print(f"Mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

  0%|          | 0/100 [00:00<?, ?it/s]

Mean_reward=1.00 +/- 0.00


In [14]:
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.repocard import metadata_eval_result, metadata_save

from pathlib import Path
import datetime
import json

## 6 - Model Deployment & Hub Integration 🚀 <a id='6'></a>

👨‍🏫 Student Insight: Sharing your Progress

By pushing your model to the Hugging Face Hub, you contribute to the open RL community. Your repository will include the Q-Table (.pkl file), the results, and a video replay of your agent successfully reaching the frisbee!

In [15]:
def record_video(env, Qtable, out_directory, fps=1):
    """
    Generate a replay video of the agent
    :param env
    :param Qtable: Qtable of our agent
    :param out_directory
    :param fps: how many frame per seconds (with taxi-v3 and frozenlake-v1 we use 1)
    """
    images = []
    terminated = False
    truncated = False
    state, info = env.reset(seed=random.randint(0, 500))
    img = env.render()
    images.append(img)
    while not terminated or truncated:
        # Take the action (index) that have the maximum expected future reward given that state
        action = np.argmax(Qtable[state][:])
        state, reward, terminated, truncated, info = env.step(
            action
        )  # We directly put next_state = state for recording logic
        img = env.render()
        images.append(img)
    imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)

In [16]:
def push_to_hub(repo_id, model, env, video_fps=1, local_repo_path="hub"):
    """
    Evaluate, Generate a video and Upload a model to Hugging Face Hub.
    This method does the complete pipeline:
    - It evaluates the model
    - It generates the model card
    - It generates a replay video of the agent
    - It pushes everything to the Hub

    :param repo_id: repo_id: id of the model repository from the Hugging Face Hub
    :param env
    :param video_fps: how many frame per seconds to record our video replay
    (with taxi-v3 and frozenlake-v1 we use 1)
    :param local_repo_path: where the local repository is
    """
    _, repo_name = repo_id.split("/")

    eval_env = env
    api = HfApi()

    # Step 1: Create the repo
    repo_url = api.create_repo(
        repo_id=repo_id,
        exist_ok=True,
    )

    # Step 2: Download files
    repo_local_path = Path(snapshot_download(repo_id=repo_id))

    # Step 3: Save the model
    if env.spec.kwargs.get("map_name"):
        model["map_name"] = env.spec.kwargs.get("map_name")
        if env.spec.kwargs.get("is_slippery", "") == False:
            model["slippery"] = False

    # Pickle the model
    with open((repo_local_path) / "q-learning.pkl", "wb") as f:
        pickle.dump(model, f)

    # Step 4: Evaluate the model and build JSON with evaluation metrics
    mean_reward, std_reward = evaluate_agent(
        eval_env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"]
    )

    evaluate_data = {
        "env_id": model["env_id"],
        "mean_reward": mean_reward,
        "n_eval_episodes": model["n_eval_episodes"],
        "eval_datetime": datetime.datetime.now().isoformat(),
    }

    # Write a JSON file called "results.json" that will contain the
    # evaluation results
    with open(repo_local_path / "results.json", "w") as outfile:
        json.dump(evaluate_data, outfile)

    # Step 5: Create the model card
    env_name = model["env_id"]
    if env.spec.kwargs.get("map_name"):
        env_name += "-" + env.spec.kwargs.get("map_name")

    if env.spec.kwargs.get("is_slippery", "") == False:
        env_name += "-" + "no_slippery"

    metadata = {}
    metadata["tags"] = [env_name, "q-learning", "reinforcement-learning", "custom-implementation"]

    # Add metrics
    eval = metadata_eval_result(
        model_pretty_name=repo_name,
        task_pretty_name="reinforcement-learning",
        task_id="reinforcement-learning",
        metrics_pretty_name="mean_reward",
        metrics_id="mean_reward",
        metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
        dataset_pretty_name=env_name,
        dataset_id=env_name,
    )

    # Merges both dictionaries
    metadata = {**metadata, **eval}

    model_card = f"""
  # **Q-Learning** Agent playing1 **{env_id}**
  This is a trained model of a **Q-Learning** agent playing **{env_id}** .

  ## Usage

  model = load_from_hub(repo_id="{repo_id}", filename="q-learning.pkl")

  # Don't forget to check if you need to add additional attributes (is_slippery=False etc)
  env = gym.make(model["env_id"])
  """

    evaluate_agent(env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"])

    readme_path = repo_local_path / "README.md"
    readme = ""
    print(readme_path.exists())
    if readme_path.exists():
        with readme_path.open("r", encoding="utf8") as f:
            readme = f.read()
    else:
        readme = model_card

    with readme_path.open("w", encoding="utf-8") as f:
        f.write(readme)

    # Save our metrics to Readme metadata
    metadata_save(readme_path, metadata)

    # Step 6: Record a video
    video_path = repo_local_path / "replay.mp4"
    record_video(env, model["qtable"], video_path, video_fps)

    # Step 7. Push everything to the Hub
    api.upload_folder(
        repo_id=repo_id,
        folder_path=repo_local_path,
        path_in_repo=".",
    )

    print("Your model is pushed to the Hub. You can view your model here: ", repo_url)

In [18]:
from huggingface_hub import notebook_login

notebook_login()

In [19]:
model = {
    "env_id": env_id,
    "max_steps": max_steps,
    "n_training_episodes": n_training_episodes,
    "n_eval_episodes": n_eval_episodes,
    "eval_seed": eval_seed,
    "learning_rate": learning_rate,
    "gamma": gamma,
    "max_epsilon": max_epsilon,
    "min_epsilon": min_epsilon,
    "decay_rate": decay_rate,
    "qtable": Qtable_frozenlake,
}

In [20]:
username = "hellosara"  # FILL THIS
repo_name = "q-FrozenLake-v1-4x4-noSlippery"
push_to_hub(repo_id=f"{username}/{repo_name}", model=model, env=env)

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

False


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...fc17b56c27/q-learning.pkl: 100%|##########|   915B /   915B            

Your model is pushed to the Hub. You can view your model here:  https://huggingface.co/hellosara/q-FrozenLake-v1-4x4-noSlippery


---

# 🎉 Congratulations! You've Mastered qlearning! <a id='7'></a>

<div style="text-align:center;">
    <img src="https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExNHJueXZueXpueXpueXpueXpueXpueXpueXpueXpueXpueXpueCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/3o7abKhOpu0NwenH3O/giphy.gif" width="400" style="border-radius: 10px;"/>
</div>

### 🎓 You are now a Reinforcement Learning Practitioner!

By completing this lab, you have successfully:
1.  **Navigated Discrete Spaces**: You mastered the grid-world logic of **FrozenLake-v1**.
2.  **Built a Tabular Brain**: You implemented a **Q-Table** from scratch without relying on complex deep learning libraries.
3.  **Balanced the Scale**: You fine-tuned the **Exploration-Exploitation** trade-off using Epsilon-Greedy decay.
4.  **Verified Perfection**: You achieved a perfect **1.00 reward**, proving your agent is a master of its environment.



### 🛰️ The Big Picture: LunarLander vs. FrozenLake

| Feature | FrozenLake (This Lab) | LunarLander (Previous Lab) |
| :--- | :--- | :--- |
| **State Space** | **Discrete** (16 specific tiles) | **Continuous** (Coordinates, velocity) |
| **Action Space** | **Discrete** (4 directions) | **Discrete** (4 engine modes) |
| **Algorithm** | **Q-Learning** (Tabular) | **PPO** (Deep Neural Networks) |
| **Complexity** | Simple enough for a lookup table | Requires a "Brain" to generalize |

---

### 🚀 What's Next?
Don't stop here! Why not try to:
* Turn on **`is_slippery=True`** and see if Q-Learning can still reach the goal.
* Change the map to **`8x8`** and see how much longer the training takes.
* Share your Hugging Face repository link in the course Discord!

**Prepared with ❄️ by Latreche Sara** *Deep Reinforcement Learning Class of 2026*